In [ ]:
"""
WarpX interactive plotter — receiving_plane vs FieldProbe + fields_sliced XY view
==================================================================================
2×2 subplot layout:
  top-left:     Ez on the receiving Y-Z port (openPMD receiving_plane)
  top-right:    TE10 reference mode shape
  bottom-left:  Ez FieldProbe N×N grid (recv_probe.dat)
  bottom-right: Ez XY slice through the domain (fields_sliced) + eb_covered overlay

S21 normalisation: incident modal amplitude computed analytically from the
injected Gaussian pulse propagated at the TE10 group velocity.

Dependencies: openpmd-api, numpy, matplotlib, scipy, ipywidgets, ipympl
  pip install openpmd-api matplotlib scipy ipywidgets ipympl
"""

# ── User settings ────────────────────────────────────────────────────────────
DIAG_DIR        = "./diags"
RECV_PLANE_NAME = "receiving_plane"
FIELDPROBE_FILE = "./diags/recv_probe.dat"
SLICED_PATH     = "./diags/fields_sliced/openpmd.bp5/"
SLICED_FIELD    = "E"
SLICED_COORD    = "z"    # Ez
SLICED_SLICE_AX = 2      # slice along Z (axis 2) → XY plane

# Waveguide geometry (WR15 — must match simulation script)
a_wg          = 3.7592e-3   # m — broad wall (Y)
b_wg          = 1.8796e-3   # m — narrow wall (Z)
recv_y_center = 0.030000    # m
recv_z_center = 0.037000    # m

# Emitter / receiver X positions
emit_x = -0.137189          # m
recv_x = +0.137189          # m

# Drive parameters (must match simulation script)
freq         = 67e9         # Hz
E0           = 1e6          # V/m — peak Ez of injected TE10
pulse_t_peak = 0.13e-9      # s
pulse_fwhm   = 0.05e-9      # s

# Physics constants
c   = 2.99792458e8
mu0 = 4e-7 * 3.141592653589793
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
from scipy.integrate import simpson
import openpmd_api as io
import os
import ipywidgets as widgets
from IPython.display import display

# requires ipympl; use %matplotlib notebook for classic Jupyter
%matplotlib widget

# ── Derived waveguide quantities ──────────────────────────────────────────────

omega   = 2 * np.pi * freq
k0      = omega / c
fc_te10 = c / (2 * a_wg)
beta    = np.sqrt(k0**2 - (np.pi / a_wg)**2)
vg      = c * np.sqrt(1 - (fc_te10 / freq)**2)
Z_TE10  = omega * mu0 / beta

te10_norm     = a_wg * b_wg / 2.0
L_propagation = abs(recv_x - emit_x)
t_arrival     = pulse_t_peak + L_propagation / vg
pulse_tau     = pulse_fwhm / (2 * np.sqrt(np.log(2)))


def incident_modal_amplitude(t):
    return E0 * te10_norm * np.exp(-(((t - t_arrival) / pulse_tau) ** 2))


def s21_from_overlap(a_recv_overlap, t):
    A_inc = incident_modal_amplitude(t)
    if abs(A_inc) < 1e-30:
        return complex(0)
    return a_recv_overlap / A_inc


# ── Helpers ──────────────────────────────────────────────────────────────────

def te10_mode(y, z, y_c, z_c, a, b):
    y_rel = y - y_c + a / 2
    z_rel = z - z_c + b / 2
    in_ap = (y_rel >= 0) & (y_rel <= a) & (z_rel >= 0) & (z_rel <= b)
    return np.where(in_ap, np.cos(np.pi * y_rel / a), 0.0)


def mode_overlap(Ez_sim, mode, dy, dz):
    return simpson(simpson(Ez_sim * mode, dx=dz, axis=1), dx=dy)


def aperture_rect(ax):
    y0 = (recv_y_center - a_wg / 2) * 1e3
    z0 = (recv_z_center - b_wg / 2) * 1e3
    ax.add_patch(Rectangle(
        (y0, z0), a_wg * 1e3, b_wg * 1e3,
        lw=1.5, edgecolor='k', facecolor='none', ls='--', label='WR15 aperture'))


# Aperture axis limits for the three port panels
_ap_ylim = ((recv_y_center - a_wg / 2) * 1e3, (recv_y_center + a_wg / 2) * 1e3)
_ap_zlim = ((recv_z_center - b_wg / 2) * 1e3, (recv_z_center + b_wg / 2) * 1e3)


def set_aperture_limits(ax):
    ax.set_xlim(*_ap_ylim)
    ax.set_ylim(*_ap_zlim)


# ── 1. Open receiving_plane openPMD series ────────────────────────────────────

series_path = os.path.join(DIAG_DIR, RECV_PLANE_NAME, "openpmd_%T.bp5")
print(f"Opening openPMD series: {series_path}")
series    = io.Series(series_path, io.Access.read_only)
all_steps = sorted(series.iterations)

_valid = []
for i in all_steps:
    try:
        _comp = series.iterations[i].meshes["E"]["z"]
        _ext  = _comp.extent
        if all(e > 0 for e in _ext):
            _valid.append(i)
    except Exception:
        pass
series.flush()
if _valid:
    all_steps = _valid
print(f"  {len(all_steps)} iterations with data: {all_steps[0]} … {all_steps[-1]}")


def load_recv_plane(step):
    it      = series.iterations[step]
    Ez_comp = it.meshes["E"]["z"]
    raw     = Ez_comp.load_chunk()
    series.flush()
    Ez_yz = raw.mean(axis=0)          # average X cells → (Ny, Nz)
    mesh  = it.meshes["E"]
    gs    = mesh.grid_spacing
    orig  = mesh.grid_global_offset
    offs  = Ez_comp.position
    ny, nz = Ez_yz.shape
    y_arr  = orig[1] + offs[1]*gs[1] + np.arange(ny) * gs[1]
    z_arr  = orig[2] + offs[2]*gs[2] + np.arange(nz) * gs[2]
    return Ez_yz, y_arr, z_arr, float(gs[1]), float(gs[2]), float(it.time)


# ── 2. Load FieldProbe file ───────────────────────────────────────────────────

print(f"Loading FieldProbe: {FIELDPROBE_FILE}")
fp_available = False
N_fp = 50
try:
    n_header = 0
    with open(FIELDPROBE_FILE) as fh:
        for ln in fh:
            if ln.lstrip().startswith(('#', '[')):
                n_header += 1
            else:
                break
    fp_all       = np.genfromtxt(FIELDPROBE_FILE, skip_header=n_header)
    fp_steps_all = fp_all[:, 0].astype(int)
    N_fp         = int(np.round(np.sqrt(np.sum(fp_steps_all == fp_steps_all[0]))))
    fp_available = True
    print(f"  FieldProbe grid: {N_fp}×{N_fp},  {len(np.unique(fp_steps_all))} steps")
except Exception as e:
    print(f"  Could not load FieldProbe ({e}) — skipping.")


def get_fp_ez(step):
    if not fp_available:
        return None
    mask = fp_steps_all == step
    if not mask.any():
        return None
    rows = fp_all[mask]
    return (rows[:, 3].reshape(N_fp, N_fp),
            rows[:, 4].reshape(N_fp, N_fp),
            rows[:, 7].reshape(N_fp, N_fp))


# ── 3. Load fields_sliced series eagerly (read_linear) ───────────────────────

print(f"Loading fields_sliced: {SLICED_PATH}")
sliced_steps    = []
sliced_data     = {}
sliced_times    = {}
sliced_extent   = {}
eb_overlay      = None
_eb_cmap        = mcolors.LinearSegmentedColormap.from_list(
    'eb_mask', [(0, 0, 0, 0), (0, 0, 0, 1)])
sliced_available = False

try:
    _series_s = io.Series(SLICED_PATH, io.Access.read_linear)
    for _it in _series_s.read_iterations():
        _i    = _it.iteration_index
        sliced_steps.append(_i)
        _mesh = _it.meshes[SLICED_FIELD]
        _comp = _mesh[SLICED_COORD]
        _gs   = _mesh.grid_spacing
        _orig = _mesh.grid_global_offset
        _raw  = _comp.load_chunk()
        _it.series_flush()
        _raw  = _raw * _comp.unit_SI
        # Average the 2 cells WarpX writes in the slice direction → (Nx, Ny)
        _arr2d = _raw.mean(axis=SLICED_SLICE_AX)
        sliced_data[_i]   = _arr2d
        sliced_times[_i]  = float(_it.time)
        # Physical extent: remaining axes are 0=X, 1=Y
        _x0 = _orig[0]; _x1 = _x0 + _gs[0] * _arr2d.shape[0]
        _y0 = _orig[1]; _y1 = _y0 + _gs[1] * _arr2d.shape[1]
        sliced_extent[_i] = [_x0, _x1, _y0, _y1]
        # eb_covered — static geometry, load once, also average slice axis
        if eb_overlay is None and "eb_covered" in _it.meshes:
            _eb   = _it.meshes["eb_covered"][io.Mesh_Record_Component.SCALAR]
            _eb_r = _eb.load_chunk()
            _it.series_flush()
            eb_overlay = _eb_r.mean(axis=SLICED_SLICE_AX)
    del _series_s
    print(f"  {len(sliced_steps)} sliced steps: {sliced_steps[0]} … {sliced_steps[-1]}")
    sliced_times_arr = np.array([sliced_times[i] for i in sliced_steps])
    sliced_available = True
except Exception as e:
    print(f"  Could not load fields_sliced ({e}) — skipping XY panel.")


# ── 4. Global colour scale ────────────────────────────────────────────────────

_sample_idx = np.linspace(0, len(all_steps)-1, min(5, len(all_steps)), dtype=int)
_vmaxes = []
for _si in _sample_idx:
    _Ez, *_ = load_recv_plane(all_steps[_si])
    _vmaxes.append(np.max(np.abs(_Ez)))
VMAX_GLOBAL = max(max(_vmaxes), 1.0)
print(f"  Colour scale: ±{VMAX_GLOBAL:.3e} V/m")

Ez0, y_arr, z_arr, dy, dz, _ = load_recv_plane(all_steps[0])
YY, ZZ  = np.meshgrid(y_arr, z_arr, indexing='ij')
mode_yz = te10_mode(YY, ZZ, recv_y_center, recv_z_center, a_wg, b_wg)


# ── 5. update_sliced (defined before update so it is always in scope) ─────────

def update_sliced(sliced_idx):
    if not sliced_available:
        return
    si   = sliced_steps[sliced_idx]
    arr  = sliced_data[si]
    t_sl = sliced_times[si]
    vmax = max(np.max(np.abs(arr)), 1.0)
    im4.set_data(arr)
    im4.set_clim(-vmax, vmax)
    title4.set_text(f"Ez XY slice  step {si}  t = {t_sl*1e9:.4f} ns")


# ── 6. Figure layout ─────────────────────────────────────────────────────────

CMAP = 'RdBu_r'
fig  = plt.figure(figsize=(12, 9))
gs_fig = gridspec.GridSpec(2, 2, figure=fig, wspace=0.4, hspace=0.45)

ax1 = fig.add_subplot(gs_fig[0, 0])   # top-left:     receiving_plane Ez
ax2 = fig.add_subplot(gs_fig[0, 1])   # top-right:    TE10 reference mode
ax3 = fig.add_subplot(gs_fig[1, 0])   # bottom-left:  FieldProbe
ax4 = fig.add_subplot(gs_fig[1, 1])   # bottom-right: XY slice

# Panel 1 — receiving_plane
im1 = ax1.pcolormesh(y_arr*1e3, z_arr*1e3, Ez0.T,
                     shading='auto', cmap=CMAP,
                     vmin=-VMAX_GLOBAL, vmax=VMAX_GLOBAL)
plt.colorbar(im1, ax=ax1, label="Ez (V/m)")
aperture_rect(ax1)
set_aperture_limits(ax1)
ax1.set_xlabel("Y (mm)"); ax1.set_ylabel("Z (mm)")
ax1.set_aspect('equal')
ax1.legend(fontsize=8, loc='upper right')
title1 = ax1.set_title("")

# Panel 2 — TE10 reference mode (static)
im2 = ax2.pcolormesh(y_arr*1e3, z_arr*1e3, mode_yz.T,
                     shading='auto', cmap=CMAP, vmin=-1, vmax=1)
plt.colorbar(im2, ax=ax2, label="Normalised")
aperture_rect(ax2)
set_aperture_limits(ax2)
ax2.set_xlabel("Y (mm)"); ax2.set_ylabel("Z (mm)")
ax2.set_title("TE10 reference mode")
ax2.set_aspect('equal')

# Panel 3 — FieldProbe
_fp0 = get_fp_ez(all_steps[0])
if _fp0 is not None:
    _fp_y0, _fp_z0, _fp_Ez0 = _fp0
else:
    _fp_y0 = np.full((N_fp, N_fp), recv_y_center)
    _fp_z0 = np.full((N_fp, N_fp), recv_z_center)
    _fp_Ez0 = np.zeros((N_fp, N_fp))
im3 = ax3.pcolormesh(_fp_y0*1e3, _fp_z0*1e3, _fp_Ez0,
                     shading='auto', cmap=CMAP,
                     vmin=-VMAX_GLOBAL, vmax=VMAX_GLOBAL)
plt.colorbar(im3, ax=ax3, label="Ez (V/m)")
aperture_rect(ax3)
set_aperture_limits(ax3)
ax3.set_xlabel("Y (mm)"); ax3.set_ylabel("Z (mm)")
ax3.set_aspect('equal')
title3 = ax3.set_title("Ez — FieldProbe" if fp_available else "FieldProbe (unavailable)")

# Panel 4 — XY slice
if sliced_available:
    _s0   = sliced_steps[0]
    _ext0 = sliced_extent[_s0]
    # imshow extent: [left, right, bottom, top] = [ymin, ymax, xmin, xmax]
    _iext0  = [_ext0[2], _ext0[3], _ext0[0], _ext0[1]]
    _vmax_s = max(np.max(np.abs(sliced_data[_s0])), 1.0)
    im4 = ax4.imshow(sliced_data[_s0], extent=_iext0, origin='lower',
                     aspect='equal', cmap=CMAP, vmin=-_vmax_s, vmax=_vmax_s)
    plt.colorbar(im4, ax=ax4, label="Ez (V/m)", shrink=0.6)
    if eb_overlay is not None:
        ax4.imshow(eb_overlay, extent=_iext0, origin='lower',
                   aspect='equal', cmap=_eb_cmap,
                   vmin=0, vmax=1, interpolation='nearest')
    ax4.set_xlabel("Y (m)"); ax4.set_ylabel("X (m)")
    ax4.set_aspect('equal')
    title4 = ax4.set_title("")
else:
    ax4.set_title("XY slice (unavailable)")

suptitle = fig.suptitle("", fontsize=12, y=1.01)
fig.tight_layout()


# ── 7. Update functions ───────────────────────────────────────────────────────

def update(step_idx):
    step = all_steps[step_idx]
    Ez_yz, _, _, dy_s, dz_s, t_sim = load_recv_plane(step)

    a_recv = mode_overlap(Ez_yz, mode_yz, dy_s, dz_s)
    s21    = s21_from_overlap(a_recv, t_sim)
    mag    = abs(s21)
    mag_dB = 20 * np.log10(max(mag, 1e-30))

    im1.set_array(Ez_yz.T.ravel())
    title1.set_text(f"Ez — receiving_plane\nt = {t_sim*1e9:.4f} ns  step {step}")

    if sliced_available:
        nearest_idx = int(np.argmin(np.abs(sliced_times_arr - t_sim)))
        update_sliced(nearest_idx)

    if fp_available:
        fp = get_fp_ez(step)
        if fp:
            fp_Ez   = fp[2]
            fp_vmax = max(np.max(np.abs(fp_Ez)), 1e-20)
            im3.set_clim(-fp_vmax, fp_vmax)
            im3.set_array(fp_Ez.ravel())
            title3.set_text(
                f"Ez — FieldProbe ({N_fp}×{N_fp})\nt = {t_sim*1e9:.4f} ns  step {step}")
        else:
            im3.set_clim(-1, 1)
            im3.set_array(np.zeros(N_fp * N_fp))
            title3.set_text(f"FieldProbe — step {step} not found")

    arrival_note = ""
    if t_sim < t_arrival - 2 * pulse_tau:
        arrival_note = "  ⚠ pulse not yet arrived"
    elif t_sim > t_arrival + 4 * pulse_tau:
        arrival_note = "  ⚠ pulse has passed — use FFT for S21"

    suptitle.set_text(
        f"|S21| = {mag:.4f}  ({mag_dB:+.2f} dB){arrival_note}"
        f"\n[t = {t_sim*1e9:.4f} ns,  pulse peak arrives at t ≈ {t_arrival*1e9:.4f} ns]")
    fig.canvas.draw_idle()


# ── 8. Widgets ────────────────────────────────────────────────────────────────

step_slider = widgets.IntSlider(
    value=0, min=0, max=len(all_steps)-1, step=1,
    description="Step idx:",
    continuous_update=False,
    layout=widgets.Layout(width='600px'),
    style={'description_width': '80px'},
)
step_label = widgets.Label(value=f"Iteration: {all_steps[0]}")
play_btn   = widgets.Play(
    value=0, min=0, max=len(all_steps)-1, step=1,
    interval=300, description="Play",
)
widgets.jslink((play_btn, 'value'), (step_slider, 'value'))


def _on_slider(change):
    step_label.value = f"Iteration: {all_steps[change['new']]}"
    update(change['new'])


step_slider.observe(_on_slider, names='value')

display(widgets.HBox([play_btn, step_slider, step_label]))
update(0)
if sliced_available:
    update_sliced(0)

print(f"\nKey times:")
print(f"  Pulse peak injected : {pulse_t_peak*1e9:.4f} ns")
print(f"  Propagation distance: {L_propagation*1e2:.3f} cm  (vg = {vg/c:.4f} c)")
print(f"  Pulse peak arrives  : {t_arrival*1e9:.4f} ns")
print(f"  Pulse ±2τ window    : [{(t_arrival-2*pulse_tau)*1e9:.4f},"
      f" {(t_arrival+2*pulse_tau)*1e9:.4f}] ns")
print(f"  TE10 norm ∫∫φ² dA  : {te10_norm:.4e} m²")
print("Ready — drag slider or press ▶")